# LightGBM

Un seul modèle pour les 3 arcs (`axe` est une feature). À tester : hyperparamètres, `features="toutes"` dans `split`, un modèle par arc, cible en ratio…

In [10]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, FEATURES

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
import lightgbm as lgb

# meilleurs réglages trouvés à la main sur le backtest débit (90.8 contre 92.5 par défaut)
PARAMS = dict(n_estimators=1000, learning_rate=0.05, num_leaves=127, objective="huber", alpha=100)  # <- à changer à la main

def modele(X_train, y_train, X_test):
    return lgb.LGBMRegressor(verbose=-1, random_state=0, **PARAMS).fit(X_train, y_train).predict(X_test)

In [12]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

                modele  naif_lag7j
axe                               
Champs-Elysees   131.0       266.5
Convention        60.3        90.3
Saints-Peres      41.4        72.3


In [ ]:
# Les 3 tests communs, contre le naïf lag_7j : semaine de Lucie (07/11/2025), test final, backtest sur TOUS les vendredis
from evaluation import trois_tests
P = trois_tests(df, modele, "lightgbm", CIBLE)

In [ ]:
# Entraîner seulement à partir d'une année : erreur sur toutes les ORIGINES, par arc et par type de jour
# (les features restent calculées sur tout l'historique : lag_an, profils… ; seules les heures d'entraînement changent)
ANNEES = [2014, 2016, 2018, 2020, 2022, 2023, 2024]  # début du train : 1er janvier de cette année
err = []
for t0 in ORIGINES:
    X_train, X_test, y_train, y_test = split(df, t0, cible=CIBLE)
    if X_test.empty:
        continue
    jour = np.select([X_test.ferie == 1, X_test.jour_sem >= 5], ["ferie", "w-e"], "semaine")
    an_train = df.datetime.loc[X_train.index].dt.year  # année de chaque heure du train
    for an in ANNEES:
        p = modele(X_train[an_train >= an], y_train[an_train >= an], X_test)
        err.append(pd.DataFrame({"annee": an, "arc": X_test.axe.astype(str), "jour": jour, "e2": (y_test - p) ** 2}))
E = pd.concat(err)
rmse_par = lambda *g: (E.groupby(["annee", *g]).e2.mean() ** .5).unstack(list(g)).round(1)  # RMSE sur toutes les heures de val
print(rmse_par("arc").assign(moyenne=lambda t: t.mean(axis=1).round(1)), "\n")
print(rmse_par("arc", "jour").to_string(), "\n")
print("heures de val par type de jour :", E[E.annee == ANNEES[0]].jour.value_counts().to_dict())